<a href="https://colab.research.google.com/github/jaqchien-dotcom/Jacob1-TW/blob/main/05_%E8%87%AA%E5%8B%95%E5%8C%96%E8%88%87%E5%A4%A7%E8%A6%8F%E6%A8%A1%E6%95%B8%E6%93%9A%E6%A6%82%E5%BF%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 自動化與大規模數據概念

黃彬華編撰

## Python 多執行緒基礎概念

Python 多執行緒 (multithreading) 是一種讓程式能同時執行多個任務的技術，能大幅提升特定情境下的執行效率。簡而言之，這種技術允許在一個主程序 (process) 中，建立多個獨立運作的執行緒 (thread)，同時處理不同的工作，讓程式看起來像是在同一時間進行多項操作。示意圖如下：

```
程式 (Process)
│
├── 主執行緒 (Main Thread)
│      ├── 執行 main()
│      └── 建立其他執行緒
│
├── 執行緒 A
│      └── 下載圖片
│
├── 執行緒 B
│      └── 存取檔案
```

Python（以最常見的 CPython 為例）有一個重要限制，就是 GIL (Global Interpreter Lock，全域直譯器鎖)，它的限制就是：同一個 Python 行程內，一次只有一個執行緒可以執行 Python 位元組碼。

*   CPU 密集型任務：因為 GIL 會阻止多個執行緒同時執行。這種情況下，多執行緒甚至可能比單執行緒執行得更慢，因為增加了切換執行緒的開銷。所以 Python 的多執行緒不適合 CPU 密集型應用 (大量數學運算、AI 計算、影像處理)，需要改用多進程 (multiprocessing) 技術。

*   I/O 密集型任務：GIL 會在執行緒等待 I/O 操作時釋放。這使得其他執行緒可以在等待期間執行，從而實現並行效果。因此，Python 的多執行緒非常適合 I/O 密集型應用（例如：檔案讀寫、網路請求）。

| <font size="4">比較項目</font> | <font size="4">多執行緒 (Thread)</font> | <font size="4">多進程 (Process)</font> |
| -------- | -------- | -------- |
| <font size="4">執行單位</font> | <font size="4">同一個 Process 內的多個 Thread</font> | <font size="4">多個獨立 Process</font> |
| <font size="4">記憶體</font> | <font size="4">共享同一塊記憶體</font> | <font size="4">各自擁有獨立記憶體</font> |
| <font size="4">建立成本</font> | <font size="4">較低</font> | <font size="4">較高</font> |
| <font size="4">切換成本</font> | <font size="4">較低</font> | <font size="4">較高</font> |
| <font size="4">資料交換</font> | <font size="4">容易，可直接共享變數</font> | <font size="4">較麻煩，需要 Queue、Pipe 等</font> |
| <font size="4">CPU 平行運算</font> | <font size="4">在 CPython 中受 GIL 限制</font> | <font size="4">可充分利用多核心 CPU</font> |
| <font size="4">適合工作</font> | <font size="4">I/O 密集</font> | <font size="4">CPU 密集</font> |



### 建立執行緒

Python 內建了 threading 模組，能快速建立和啟動執行緒。

In [ ]:
import threading
import time

# 每個執行緒要執行的工作
# name, delay_time 為建立 Thread 時透過 args 傳入的參數
def task(name, delay_time):
    for i in range(1, 6):
        print(f"{name} 跑第 {i} 圈")
        # 讓執行緒暫停指定秒數
        time.sleep(delay_time)


def main():
    # 建立執行緒物件(但尚未執行)
    #
    # target
    # 指定新執行緒啟動後要執行的函式
    #
    # args
    # 傳遞給 target 函式的參數。型別為 tuple，
    # 所以即便只有一個值，後面逗號不可省略，例如：args=("John",)
    john = threading.Thread(
        target=task,
        args=("John", 0.1)
    )

    mary = threading.Thread(
        target=task,
        args=("Mary", 0)
    )

    # start() 會啟動 john 執行緒，並執行 target 所指定的函式
    john.start()

    mary.start()

    # 主執行緒等待 john 執行完成。
    # 若沒有 join()，主執行緒可能先結束程式。
    john.join()

    # 等待 mary 執行完成
    mary.join()

    print("全部完成")


main()

John 跑第 1 圈
Mary 跑第 1 圈
Mary 跑第 2 圈
Mary 跑第 3 圈
Mary 跑第 4 圈
Mary 跑第 5 圈
John 跑第 2 圈
John 跑第 3 圈
John 跑第 4 圈
John 跑第 5 圈
全部完成


### 執行緒安全性與鎖定機制

多執行緒會「共享同一份資料」，所以如果多個執行緒同時修改同一個變數，就可能發生 Race Condition 錯誤。屬於執行緒安全問題。

使用 Lock 解決：Lock 可以讓某段程式同一時間只允許一個執行緒進入，其他執行緒必須在外面等待，直到前一個執行緒出來方可進入。

*下方範例未使用 Lock*

```
Race Condition 示意圖

John 帳戶：
1000
 │
 ├──────────────┐
 │              │
 │              │
P 公司        U 公司
 │              │
讀到1000      讀到1000
 │              │
+500          +800
 │              │
1500          1800
 │              │
 └──────┬───────┘
        │
後計算會蓋掉前計算的值
```

In [ ]:
import threading
import time


# John 的帳戶餘額
balance = 1000


def transfer(company, amount):
    global balance

    print(f"{company} 開始匯款 {amount} 元")

    # 模擬讀取帳戶餘額
    current_balance = balance

    # 模擬銀行系統處理時間
    time.sleep(1)

    # 更新帳戶餘額
    balance = current_balance + amount

    print(f"{company} 匯款完成，目前餘額：{balance}")


def main():
    print("未使用 Lock\n")

    t1 = threading.Thread(
        target=transfer,
        args=("P 公司", 500)
    )

    t2 = threading.Thread(
        target=transfer,
        args=("U 公司", 800)
    )

    t1.start()
    t2.start()

    t1.join()
    t2.join()

    print("-" * 30)
    print(f"John 最終餘額：{balance}")


main()

未使用 Lock

P 公司 開始匯款 500 元
U 公司 開始匯款 800 元
P 公司 匯款完成，目前餘額：1500
U 公司 匯款完成，目前餘額：1800
------------------------------
John 最終餘額：1800


*下方範例使用 Lock*

```
流程變成：

John

1000
 │
 │
P 公司取得 Lock
 │
1000 + 500
 │
1500
 │
釋放 Lock
 │
 │
U 公司取得 Lock
 │
1500 + 800
 │
2300

最後一定為 2300 元
```

In [ ]:
import threading
import time


balance = 1000

# 建立 Lock
lock = threading.Lock()


def transfer(company, amount):
    global balance

    print(f"{company} 開始匯款 {amount} 元")

    # 同一時間只允許一家公司修改帳戶
    with lock:
        current_balance = balance

        # 模擬銀行處理時間
        time.sleep(1)

        balance = current_balance + amount

        print(f"{company} 匯款完成，目前餘額：{balance}")


def main():
    print("使用 Lock\n")

    t1 = threading.Thread(
        target=transfer,
        args=("P 公司", 500)
    )

    t2 = threading.Thread(
        target=transfer,
        args=("U 公司", 800)
    )

    t1.start()
    t2.start()

    t1.join()
    t2.join()

    print("-" * 30)
    print(f"John 最終餘額：{balance}")


main()

使用 Lock

P 公司 開始匯款 500 元
U 公司 開始匯款 800 元
P 公司 匯款完成，目前餘額：1500
U 公司 匯款完成，目前餘額：2300
------------------------------
John 最終餘額：2300


### 執行緒池概念

當任務數量過多時，手動建立、啟動與等待每一個執行緒會非常繁瑣。這時可以使用 concurrent.futures 套件中的 ThreadPoolExecutor，它會預先建立一組執行緒並放入執行緒池 (Thread Pool)，需要工作時直接從池中取出執行緒執行，而不必每次都建立新的執行緒。它還會自動幫我們管理執行緒的生命週期與數量，這比使用 threading 模組手動建立大量執行緒更方便，也更適合實務開發。
```
執行緒池執行流程

Main Thread
      │
      │
建立 ThreadPool (2 個 Thread)
      │
      ├──────────────┐
      │              │
submit()         submit()
      │              │
Thread 1         Thread 2
      │              │
 task()          task()
      │              │
      └───────完成────┘
             │
        離開 with
             │
        全部完成
```

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time


# 每個執行緒要執行的工作
def task(name, delay_time):
    print(f"{name} 開始執行")
    time.sleep(delay_time)
    print(f"{name} 執行完成")


def main():
    # 建立執行緒池
    # max_workers=2，表示最多同時只有 2 個執行緒工作
    with ThreadPoolExecutor(max_workers=2) as executor:
        # 提交第一個工作
        executor.submit(task, "John", 1)
        # 提交第二個工作
        executor.submit(task, "Mary", 2)

    # 離開 with 時，
    # ThreadPoolExecutor 會自動等待所有工作完成
    print("全部完成")


main()

John 開始執行
Mary 開始執行
John 執行完成
Mary 執行完成
全部完成


### 執行緒池應用

Python 的 ThreadPoolExecutor 會自動管理執行緒的建立、重複利用、等待完成與釋放資源，我們只要專心提交工作 (submit) 即可。可應用在網頁爬蟲、API 批次呼叫、檔案下載等情境。



In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random


# 模擬爬取單一網頁
# url：欲爬取的網址
# 回傳 dict：包含網址與耗時
def fetch_page(url):
    print(f"開始爬取：{url}")

    # 下方程式模擬爬蟲花費時間 (1~3 秒)，可替換成真正的爬蟲程式 (requests + BeautifulSoup)
    delay = random.randint(1, 3)
    time.sleep(delay)

    # 模擬解析 HTML 後取得的資料
    result = {
        "url": url,
        "delay": delay
    }

    print(f"完成爬取：{url}")

    # 將爬取結果回傳給呼叫者
    return result


def main():

    # 欲爬取的網址清單
    urls = [
        "https://example.com/page1",
        "https://example.com/page2",
        "https://example.com/page3",
        "https://example.com/page4",
        "https://example.com/page5"
    ]

    # 儲存所有爬取結果
    results = []

    # 建立執行緒池，max_workers=3 表示最多建立 3 個執行緒。
    # 即使有 100 個網址，也只會有最多 3 個執行緒同時工作。
    # with 結束後，ThreadPoolExecutor 會自動等待所有工作完成，並釋放所有執行緒。
    with ThreadPoolExecutor(max_workers=3) as executor:

        # 建立 dict 存放：
        # Key：Future 物件；Value：對應的網址
        # 方便之後知道每個 future 對應的網址
        future_to_url = {}

        # 將所有網址交給執行緒池
        for url in urls:
            # 將 fetch_page 工作以及所需 URL 提交給執行緒池。
            future = executor.submit(fetch_page, url)
            # 記錄 Future 與網址的對應關係
            future_to_url[future] = url

        # concurrent.futures.as_completed(fs, timeout=None)
        # 其中 fs 可以是儲存著 Future 物件的集合，例如：list[Future]，
        # 而 dict 則 key 需為 Future 物件。
        # 依照完成順序逐一回傳 Future 物件，不一定與 urls 原始順序一致。
        for future in as_completed(future_to_url):
            # 找出是哪一個網址完成
            url = future_to_url[future]
            try:
                # result()取得執行緒的回傳值，若工作尚未完成，會等待直到完成。
                result = future.result()
                # 加入結果清單
                results.append(result)
            except Exception as e:
                # 若某個執行緒發生例外，不會影響其他執行緒。
                print(f"{url} 爬取失敗：{e}")

    print("全部爬取完成")
    print("-" * 30)

    # 顯示所有爬取結果
    for result in results:
        print(f"網址：{result['url']}")
        print(f"耗時：{result['delay']} 秒")
        print("-" * 30)


main()

開始爬取：https://example.com/page1
開始爬取：https://example.com/page2
開始爬取：https://example.com/page3
完成爬取：https://example.com/page1
開始爬取：https://example.com/page4
完成爬取：https://example.com/page2
開始爬取：https://example.com/page5
完成爬取：https://example.com/page3
完成爬取：https://example.com/page4
完成爬取：https://example.com/page5
全部爬取完成
------------------------------
網址：https://example.com/page1
耗時：2 秒
------------------------------
網址：https://example.com/page2
耗時：2 秒
------------------------------
網址：https://example.com/page3
耗時：2 秒
------------------------------
網址：https://example.com/page4
耗時：1 秒
------------------------------
網址：https://example.com/page5
耗時：3 秒
------------------------------


## Python 自動化排程

Python 自動化排程 (Task Scheduling) 是讓程式依照指定的時間或條件，自動執行某些工作，例如：

* 每天凌晨備份資料
* 每小時抓取網站資料
* 每週寄送報表
* 每月產生統計分析
* 每隔 5 分鐘監控系統狀態

這是資料工程、AI、自動化、爬蟲、系統管理都很常見的應用。

### while + time.sleep()

使用 while + time.sleep() 方式最簡明易懂，優缺點說明如下：

優點：語法極度簡單，且不需要安裝第三方套件。

缺點：
* 時間漂移（Time Drift）： sleep 的時間是「任務結束後」才開始算。如果任務每次執行需要花 10 秒，設定 sleep 60 秒，那麼實際上每次執行的間隔會變成 70 秒。時間一長，執行時間點就會完全偏離。
* 無法精準定時： 沒辦法做到「每天早上 8:30 準時執行」這種功能，它只能做到「每隔 X 秒執行一次」。
* 卡死風險： 它是單執行緒運作，如果任務卡死（例如網路請求沒有設定 timeout），整個排程程式就會永遠卡在那裡，不會再進入下一次循環。

In [ ]:
import time
from datetime import datetime

def job():
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 執行自動化任務...")
    # 在這裡放您的自動化程式碼，例如爬蟲、寄信等

# 設定執行週期的秒數（例如 300 秒 = 5 分鐘）
INTERVAL_SECONDS = 5

print("排程啟動...")
while True:
    try:
        job()
    except Exception as e:
        print(f"錯誤發生，但排程將繼續運行。錯誤原因: {e}")

    # 讓程式暫停指定的秒數
    time.sleep(INTERVAL_SECONDS)


排程啟動...
[2026-07-02 14:07:58] 執行自動化任務...
[2026-07-02 14:08:03] 執行自動化任務...


KeyboardInterrupt: 

### schedule 套件

schedule 套件幾乎解決了 while + sleep() 的所有致命缺點。它之所以被廣泛使用，正是因為它在底層幫忙處理了時間計算與定時邏輯。

In [ ]:
# 安裝 schedule 套件
!pip install schedule

In [ ]:
from datetime import datetime
import time
import schedule

def job():
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 執行自動化任務...")
    # 在這裡放您的自動化程式碼，例如爬蟲、寄信等


# 清除舊的排程工作，以避免重複執行
schedule.clear()

# 每 5 秒/分/時/日/週 執行一次 job()；every() 預設值為 1
schedule.every(5).seconds.do(job)
schedule.every(5).minutes.do(job)
schedule.every(5).hours.do(job)
schedule.every(5).days.do(job)
schedule.every(5).weeks.do(job)

# 每天 09:00 執行一次 job()
schedule.every().day.at("09:00").do(job)

# 每週一 09:00 執行一次 job()
schedule.every().monday.at("09:00").do(job)


while True:
    # 檢查是否有等待排程的工作，有則立即執行；沒有則直接返回
    schedule.run_pending()
    # 每隔 1 秒再檢查一次是否有新的排程工作需要執行，避免 while 迴圈不停執行而浪費 CPU 資源
    time.sleep(1)

[2026-07-02 14:08:24] 執行自動化任務...
[2026-07-02 14:08:29] 執行自動化任務...


KeyboardInterrupt: 

schedule 套件也可為不同工作設定排程。

In [ ]:
import schedule
from datetime import datetime
import time


def crawler():
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 開始爬蟲...")


def backup():
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 備份資料...")


# 清除舊的排程工作，以避免重複執行
schedule.clear()

schedule.every(5).seconds.do(crawler)
schedule.every(12).seconds.do(backup)

while True:
    schedule.run_pending()
    time.sleep(1)

[2026-07-02 14:08:41] 開始爬蟲...
[2026-07-02 14:08:46] 開始爬蟲...
[2026-07-02 14:08:48] 備份資料...


KeyboardInterrupt: 

## Pandas

Pandas 是 Python 程式語言中最受歡迎的資料分析與處理函式庫。你可以把它想像成「程式版的 Excel」，它結合了試算表的直覺性以及 SQL 指令強大的資料操作能力，被廣泛應用於數據科學、機器學習及金融分析等領域。



### 讀取 CSV, JSON, Excel 檔案

In [ ]:
import pandas as pd

# 讀取 CSV 檔案，並指定欄位型別
df = pd.read_csv(
    "sample_employees.csv",
    dtype={
        "ID": int,
        "Age": int,
        "Salary": int
    }
)
print("員工資料", df, sep="\n")

# 讀取 JSON 檔案
# df = pd.read_json("data.json")

# 讀取 Excel 檔案
# df = pd.read_excel("data.xlsx")


員工資料
    ID   Name  Gender  Age Department  Salary   Hire_Date
0  101   John    Male   28      Sales   45000  2024-03-15
1  102   Mary  Female   35         RD   62000  2022-08-01
2  103  David    Male   42      Sales   55000  2021-11-10
3  104   Emma  Female   31         HR   48000  2025-01-05
4  105    Ken    Male   25         RD   50000  2025-06-20


### 資料的綱要與統計資訊

In [ ]:
# 資料的綱要資訊
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          5 non-null      int64 
 1   Name        5 non-null      object
 2   Gender      5 non-null      object
 3   Age         5 non-null      int64 
 4   Department  5 non-null      object
 5   Salary      5 non-null      int64 
 6   Hire_Date   5 non-null      object
dtypes: int64(3), object(4)
memory usage: 412.0+ bytes


In [ ]:
# 資料的統計資訊
print(df[["Age", "Salary"]].describe())

             Age        Salary
count   5.000000      5.000000
mean   32.200000  52000.000000
std     6.610598   6670.832032
min    25.000000  45000.000000
25%    28.000000  48000.000000
50%    31.000000  50000.000000
75%    35.000000  55000.000000
max    42.000000  62000.000000


### 查看前後幾筆資料

In [ ]:
# 查看資料的前 2 筆 (預設為 5 筆)
print(df.head(2))
print("-" * 60)
# 查看資料的後 2 筆 (預設為 5 筆)
print(df.tail(2))

    ID  Name  Gender  Age Department  Salary   Hire_Date
0  101  John    Male   28      Sales   45000  2024-03-15
1  102  Mary  Female   35         RD   62000  2022-08-01
------------------------------------------------------------
    ID  Name  Gender  Age Department  Salary   Hire_Date
3  104  Emma  Female   31         HR   48000  2025-01-05
4  105   Ken    Male   25         RD   50000  2025-06-20


### 取出指定的欄或列資料

In [ ]:
# 取出指定欄位資料
print(df[["Name", "Age", "Gender"]])

    Name  Age  Gender
0   John   28    Male
1   Mary   35  Female
2  David   42    Male
3   Emma   31  Female
4    Ken   25    Male


In [ ]:
# 取出指定列資料：0 到 2 (包含 2)
print(df.loc[:2])

    ID   Name  Gender  Age Department  Salary   Hire_Date
0  101   John    Male   28      Sales   45000  2024-03-15
1  102   Mary  Female   35         RD   62000  2022-08-01
2  103  David    Male   42      Sales   55000  2021-11-10


In [ ]:
# 取出指定列與欄位資料
print(df.loc[:2, ["Name", "Age", "Gender"]])

    Name  Age  Gender
0   John   28    Male
1   Mary   35  Female
2  David   42    Male


### 將指定欄位設為 index

In [ ]:
# 將指定欄位設為 index
df.set_index("ID", inplace=True)
print(df)
print("-" * 60)

# 透過新 index 取值
print(df.loc[101])
print("-" * 60)

# 重設 index。「inplace=True」會直接修改原本的 DataFrame
df.reset_index(inplace=True)
print(df)

      Name  Gender  Age Department  Salary   Hire_Date
ID                                                    
101   John    Male   28      Sales   45000  2024-03-15
102   Mary  Female   35         RD   62000  2022-08-01
103  David    Male   42      Sales   55000  2021-11-10
104   Emma  Female   31         HR   48000  2025-01-05
105    Ken    Male   25         RD   50000  2025-06-20
------------------------------------------------------------
Name                John
Gender              Male
Age                   28
Department         Sales
Salary             45000
Hire_Date     2024-03-15
Name: 101, dtype: object
------------------------------------------------------------
    ID   Name  Gender  Age Department  Salary   Hire_Date
0  101   John    Male   28      Sales   45000  2024-03-15
1  102   Mary  Female   35         RD   62000  2022-08-01
2  103  David    Male   42      Sales   55000  2021-11-10
3  104   Emma  Female   31         HR   48000  2025-01-05
4  105    Ken    Male   25  

### 排序

In [ ]:
# 依照年齡排序
df.sort_values(by="Age", ascending=True, inplace=True)
print(df)

    ID   Name  Gender  Age Department  Salary   Hire_Date
4  105    Ken    Male   25         RD   50000  2025-06-20
0  101   John    Male   28      Sales   45000  2024-03-15
3  104   Emma  Female   31         HR   48000  2025-01-05
1  102   Mary  Female   35         RD   62000  2022-08-01
2  103  David    Male   42      Sales   55000  2021-11-10


### 查詢資料

In [ ]:
# 條件查詢
print("找出年齡小於等於 30 歲的員工")
print(df[df["Age"] <= 30])
print("-" * 60)

print("找出年齡小於等於 30 歲且薪資大於等於 50000 的員工")
print(df[(df["Age"] <= 30) & (df["Salary"] >= 50000)])

找出年齡小於等於 30 歲的員工
    ID  Name Gender  Age Department  Salary   Hire_Date
4  105   Ken   Male   25         RD   50000  2025-06-20
0  101  John   Male   28      Sales   45000  2024-03-15
------------------------------------------------------------
找出年齡小於等於 30 歲且薪資大於等於 50000 的員工
    ID Name Gender  Age Department  Salary   Hire_Date
4  105  Ken   Male   25         RD   50000  2025-06-20


### 新增資料

In [ ]:
# 新增一筆資料
new_employee = {
    "ID": 106,
    "Name": "Lisa",
    "Gender": "Female",
    "Age": 29,
    "Department": "HR",
    "Salary": 47000,
    "Hire_Date": "2026-07-01"
}

df = pd.concat(
    [df, pd.DataFrame([new_employee])],
    ignore_index=True
)
print(df)

    ID   Name  Gender  Age Department  Salary   Hire_Date
0  105    Ken    Male   25         RD   50000  2025-06-20
1  101   John    Male   28      Sales   45000  2024-03-15
2  104   Emma  Female   31         HR   48000  2025-01-05
3  102   Mary  Female   35         RD   62000  2022-08-01
4  103  David    Male   42      Sales   55000  2021-11-10
5  106   Lisa  Female   29         HR   47000  2026-07-01


### 修改資料

In [ ]:
# 將名為 "Lisa" 的員工薪水改為45000
df.loc[df["Name"] == "Lisa", "Salary"] = 45000
print(df)

    ID   Name  Gender  Age Department  Salary   Hire_Date
0  105    Ken    Male   25         RD   50000  2025-06-20
1  101   John    Male   28      Sales   45000  2024-03-15
2  104   Emma  Female   31         HR   48000  2025-01-05
3  102   Mary  Female   35         RD   62000  2022-08-01
4  103  David    Male   42      Sales   55000  2021-11-10
5  106   Lisa  Female   29         HR   45000  2026-07-01


### 刪除資料


In [ ]:
# 留下所有名字不是 Lisa 的資料，並直接覆寫原本的 df
df = df[df["Name"] != "Lisa"]
print(df)

    ID   Name  Gender  Age Department  Salary   Hire_Date
0  105    Ken    Male   25         RD   50000  2025-06-20
1  101   John    Male   28      Sales   45000  2024-03-15
2  104   Emma  Female   31         HR   48000  2025-01-05
3  102   Mary  Female   35         RD   62000  2022-08-01
4  103  David    Male   42      Sales   55000  2021-11-10


### 分組統計

In [ ]:
# 分組統計
print("依據部門分組，並顯示分組情形")
for group in df.groupby("Department"):
    print(group)
print("-" * 30)

print("依據部門分組，計算各部門平均薪資")
department_salary_mean = df.groupby("Department")["Salary"].mean()
print(department_salary_mean)
print("-" * 30)

print("依據性別分組，計算平均薪資")
gender_salary_mean = df.groupby('Gender')['Salary'].mean()
print(gender_salary_mean)

依據部門分組，並顯示分組情形
('HR',     ID  Name  Gender  Age Department  Salary   Hire_Date
2  104  Emma  Female   31         HR   48000  2025-01-05)
('RD',     ID  Name  Gender  Age Department  Salary   Hire_Date
0  105   Ken    Male   25         RD   50000  2025-06-20
3  102  Mary  Female   35         RD   62000  2022-08-01)
('Sales',     ID   Name Gender  Age Department  Salary   Hire_Date
1  101   John   Male   28      Sales   45000  2024-03-15
4  103  David   Male   42      Sales   55000  2021-11-10)
------------------------------
依據部門分組，計算各部門平均薪資
Department
HR       48000.0
RD       56000.0
Sales    50000.0
Name: Salary, dtype: float64
------------------------------
依據性別分組，計算平均薪資
Gender
Female    55000.0
Male      50000.0
Name: Salary, dtype: float64


### 資料合併

In [ ]:
import pandas as pd

# 讀取員工 CSV 檔案
employees = pd.read_csv(
    "sample_employees.csv",
    dtype={
        "ID": int,
        "Age": int,
        "Salary": int
    }
)
print("員工資料", employees, sep="\n")
print("-" * 60)

# 讀取員工聯絡方式 CSV 檔案
contacts = pd.read_csv(
    "sample_employees_contacts.csv",
    dtype={
        "Emp_ID": int,
        "Phone": str
    }
)
print("員工聯絡方式", contacts, sep="\n")
print("-" * 60)

# 使用 merge() 基於 ID 屬性進行合併
# merge() 的 how 參數預設為 "inner"；也可使用 "left", "right", "outer"
# 關聯欄位名稱相同：on 即可；欄位名稱不同：left_on, right_on 都需要
df_merged = employees[["ID", "Name"]].merge(
    contacts,
    left_on="ID",
    right_on="Emp_ID",
    how="inner"
)
print("合併員工資料與聯絡方式", df_merged, sep="\n")

員工資料
    ID   Name  Gender  Age Department  Salary   Hire_Date
0  101   John    Male   28      Sales   45000  2024-03-15
1  102   Mary  Female   35         RD   62000  2022-08-01
2  103  David    Male   42      Sales   55000  2021-11-10
3  104   Emma  Female   31         HR   48000  2025-01-05
4  105    Ken    Male   25         RD   50000  2025-06-20
------------------------------------------------------------
員工聯絡方式
   Emp_ID              Email       Phone
0     101   john@company.com  0912345678
1     102   mary@company.com  0923456789
2     103  david@company.com  0934567890
3     104   emma@company.com  0945678901
4     108    ben@company.com  0987654321
------------------------------------------------------------
合併員工資料與聯絡方式
    ID   Name  Emp_ID              Email       Phone
0  101   John     101   john@company.com  0912345678
1  102   Mary     102   mary@company.com  0923456789
2  103  David     103  david@company.com  0934567890
3  104   Emma     104   emma@company.com  094567

### Pandas 在大數據上的限制

* 記憶體限制 (Memory Limitation)：Pandas 採用記憶體內運算，通常需要將整個 DataFrame 載入 RAM。當資料量過大時，可能超過可用記憶體，造成 MemoryError，或是造成運算速度大幅下降。
* 單機處理 (Single-Machine Processing)：Pandas 主要設計用於單一電腦，缺乏原生的分散式運算能力，不適合處理數十、數百 GB 或 TB 級的大數據。此時通常會改用 Dask 或 Apache Spark 等支援平行與分散式運算的框架。


## Dask

Dask 是一個支援平行與分散式運算的框架，有下列特點：

* 可擴展性 (Scalability)：將大型資料集切分成多個分區 (Partition)，Dask 可以逐步讀取與處理資料，因此能夠分析遠大於 RAM 容量的資料集

* 平行運算 (Parallel Computing)：Dask 能將工作拆分成多個任務，並同時執行。可以善用單機多核心 (充分利用一台電腦的多核心 CPU)；亦能做到分散式運算 ( 將工作分配到多台電腦共同完成)。因此能有效縮短大型資料集的處理時間。

* 熟悉的 API：Dask 提供與 Pandas 高度相似的 API，例如：Dask DataFrame 類似 Pandas DataFrame。

* 惰性求值 (Lazy Evaluation)：程式執行時，不會立即開始運算，而是先建立一張任務圖 (Task Graph)；呼叫 compute()，才會開始執行所有任務，如此可以提高執行效率。

Dask 並不是要取代 Pandas，而是將 Pandas 擴展到大型資料處理與平行運算的場景。

因此：

Pandas：資料量不大，可完全放入記憶體內，建議使用 Pandas，操作更簡單、執行速度也常較快。

Dask：資料量超過記憶體大小，或希望充分利用多核心、甚至多台電腦共同運算，建議使用 Dask。

```
Dask 的處理流程示意圖：

           read_csv()
                │
                ▼
        建立 Dask DataFrame
                │
                ▼
      merge()、filter()、groupby()
                │
      (都只是建立 Task Graph)
                │
                ▼
          compute()
                │
                ▼
      真正開始讀檔、平行運算
                │
                ▼
      回傳 Pandas DataFrame
```
可以把 .compute() 理解成「開始執行」按鈕：
* dd.read_csv()：規劃工作
* merge()：再加入一項工作
* groupby()：再加入一項工作
* compute()：一次執行所有工作，並回傳 Pandas DataFrame。這也是 Dask 能夠最佳化整體工作流程、有效利用多核心或分散式叢集的原因。

In [ ]:
import dask.dataframe as dd

# 讀取 CSV 檔案 (規劃工作，尚未執行)
df = dd.read_csv(
    "sample_employees.csv",
    dtype={
        "ID": int,
        "Age": int,
        "Salary": int
    }
)

# df 資料為空，因為未呼叫 compute()，所以尚未真正執行讀取動作
print("員工資料", df, sep="\n")
print("-" * 60)

# 查詢「年齡小於等於 40 歲的員工」 (加入一項工作)
result = df[df["Age"] <= 40]

# 依照年齡排序 (再加入一項工作)
# Dask 沒有 inplace=True，必須接回新的 DataFrame
df = df.sort_values(by="Age", ascending=True)

# 呼叫 compute() 才是真正開始執行所有工作 (讀取資料 -> 查詢 -> 排序)
print(df.compute())

員工資料
Dask DataFrame Structure:
                  ID    Name  Gender    Age Department Salary Hire_Date
npartitions=1                                                          
               int64  string  string  int64     string  int64    string
                 ...     ...     ...    ...        ...    ...       ...
Dask Name: to_string_dtype, 2 expressions
Expr=ArrowStringConversion(frame=FromMapProjectable(e463fc7))
------------------------------------------------------------
    ID   Name  Gender  Age Department  Salary   Hire_Date
4  105    Ken    Male   25         RD   50000  2025-06-20
0  101   John    Male   28      Sales   45000  2024-03-15
3  104   Emma  Female   31         HR   48000  2025-01-05
1  102   Mary  Female   35         RD   62000  2022-08-01
2  103  David    Male   42      Sales   55000  2021-11-10


In [ ]:
import dask.dataframe as dd

# 讀取員工 CSV 檔案 (規劃工作，尚未執行)
employees = dd.read_csv(
    "sample_employees.csv",
    dtype={
        "ID": int,
        "Age": int,
        "Salary": int
    }
)

# 讀取員工聯絡方式 CSV 檔案 (加入一項工作)
contacts = dd.read_csv(
    "sample_employees_contacts.csv",
    dtype={
        "Emp_ID": int,
        "Phone": str
    }
)

# 使用 merge() 進行合併 (再加入一項工作)
df_merged = employees[["ID", "Name"]].merge(
    contacts,
    left_on="ID",
    right_on="Emp_ID",
    how="inner"
)

print("員工資料與聯絡方式合併")
# 呼叫 compute() 才是真正開始執行所有工作 (讀取員工資料 -> 讀取聯絡方式 -> 合併)
print(df_merged.compute())

員工資料與聯絡方式合併
    ID   Name  Emp_ID              Email       Phone
0  101   John     101   john@company.com  0912345678
1  102   Mary     102   mary@company.com  0923456789
2  103  David     103  david@company.com  0934567890
3  104   Emma     104   emma@company.com  0945678901
